# SVR Regression — K-Fold CV and Hyperparameter Tuning

Regression notebook using **Support Vector Regression (SVR)** with:

- `ColumnTransformer` for robust preprocessing (imputation + scaling + encoding) inside each CV fold
- `KFold` for cross-validated evaluation
- `GridSearchCV` with per-kernel parameter grids
- Log-transform on the target to optimise RMSLE (Kaggle's metric)

Adapt the sections marked as `TODO` (target column, features, param grid, etc.).  
Tested on the [Kaggle House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) dataset.

In [2]:
# ============================================================
# SECTION 0 - Imports and base configuration
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

In [3]:
# ============================================================
# SECTION 1 - Load dataset
# ============================================================

# TODO: replace "train.csv" with your file (csv / xlsx, etc.)
df = pd.read_csv("train.csv")  # e.g.: pd.read_excel("data.xlsx")

print(df.head())
print(df.info())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

In [4]:
# ============================================================
# SECTION 2 - Define target and features
# ============================================================

# TODO: name of the target column (continuous variable to predict)
TARGET_COL = "SalePrice"   # <--- CHANGE THIS

# TODO: columns to explicitly exclude (e.g. identifiers that carry no signal)
EXCLUDE_COLS = ["Id"]      # <--- add other columns to exclude here

# Use all columns except the target and excluded ones as features
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL and c not in EXCLUDE_COLS]

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 79)
y shape: (1460,)


In [5]:
# ============================================================
# SECTION 3 - Preprocessing pipeline (ColumnTransformer)
# ============================================================
# The ColumnTransformer is defined here but NOT fitted yet.
# It will be fitted inside each CV fold by GridSearchCV — no data leakage.

numeric_cols     = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

print(f"Numeric features   : {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

# Numeric: impute with median (robust to outliers) + standardise
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

# Categorical: impute with most frequent + one-hot encode
# handle_unknown='ignore' prevents crashes on unseen categories in the test set
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,    numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

Numeric features   : 36
Categorical features: 43


In [6]:
# ============================================================
# SECTION 4 - Train/Test split
# ============================================================
# X stays as a DataFrame — the ColumnTransformer handles everything internally.
# Log-transform the target: RMSE on log scale ≈ RMSLE (Kaggle's metric).
# Remove y_log and use y.values directly if your dataset does not use a log metric.

y_log = np.log1p(y.values)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log,
    test_size=0.2,
    random_state=SEED
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

X_train shape: (1168, 79)
X_test  shape: (292, 79)


In [7]:
# ============================================================
# SECTION 5 - SVR Pipeline + KFold + GridSearchCV
# ============================================================

# Full pipeline: ColumnTransformer + SVR
# GridSearchCV receives raw X_train (DataFrame) — preprocessing happens inside each fold.
svr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("svr",          SVR())
])

# List of dicts: one grid per kernel to avoid invalid combinations
# (e.g. gamma is irrelevant for linear kernel)
# TODO: extend or narrow the ranges based on your dataset
param_grid = [
    {
        "svr__kernel":  ["linear"],
        "svr__C":       [0.1, 1.0, 10.0, 50.0],
        "svr__epsilon": [0.01, 0.05, 0.1],
    },
    {
        "svr__kernel":  ["rbf"],
        "svr__C":       [1.0, 10.0, 50.0, 100.0],
        "svr__gamma":   ["scale", "auto", 0.01, 0.001],
        "svr__epsilon": [0.01, 0.05, 0.1],
    },
]

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=kf,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("Starting hyperparameter search...")
search.fit(X_train, y_train_log)

best_cv_rmsle = np.sqrt(-search.best_score_)
print(f"\nBest parameters : {search.best_params_}")
print(f"Best CV RMSLE   : {best_cv_rmsle:.4f}")

Starting hyperparameter search...
Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best parameters : {'svr__C': 10.0, 'svr__epsilon': 0.05, 'svr__gamma': 0.001, 'svr__kernel': 'rbf'}
Best CV RMSLE   : 0.1195


In [8]:
# ============================================================
# SECTION 6 - Evaluate best model on the holdout test set
# ============================================================

best_model  = search.best_estimator_
y_pred_log  = best_model.predict(X_test)

# Back-transform to original scale for interpretable metrics
y_pred      = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test_log)

rmsle = np.sqrt(mean_squared_error(y_test_log, y_pred_log))  # ≈ Kaggle metric
rmse  = np.sqrt(mean_squared_error(y_test_orig, y_pred))     # in target units ($)
mae   = mean_absolute_error(y_test_orig, y_pred)
r2    = r2_score(y_test_orig, y_pred)

print("=== Best Ridge — Holdout Test Set ===")
print(f"RMSLE (≈ Kaggle metric)   : {rmsle:.4f}")
print(f"RMSE  (in target units)   : {rmse:.4f}")
print(f"MAE   (in target units)   : {mae:.4f}")
print(f"R²                        : {r2:.4f}")

=== Best Ridge — Holdout Test Set ===
RMSLE (≈ Kaggle metric)   : 0.1265
RMSE  (in target units)   : 23691.6900
MAE   (in target units)   : 14098.6998
R²                        : 0.9268


In [9]:
# ============================================================
# SECTION 7 - Final training on the full dataset + Submission
# ============================================================

# Extract best SVR params and re-train on ALL data
best_svr_params = {
    k.replace("svr__", ""): v
    for k, v in search.best_params_.items()
    if k.startswith("svr__")
}

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("svr",          SVR(**best_svr_params))
])

y_log_full = np.log1p(y.values)
final_model.fit(X, y_log_full)
print(f"Final training done on {X.shape[0]} samples.")
print(f"Best parameters: {best_svr_params}")

# --- Load and predict on the test set ---
# The pipeline handles imputation, scaling and encoding automatically.
test_df  = pd.read_csv("test.csv")
test_ids = test_df["Id"]
test_X   = test_df[FEATURE_COLS].copy()

# Model predicts on log scale → expm1 to recover original SalePrice
test_preds = np.expm1(final_model.predict(test_X))

submission = pd.DataFrame({"Id": test_ids, "SalePrice": test_preds})
submission.to_csv("submission.csv", index=False)

print("Submission saved to: submission.csv")
print(submission.head())

Final training done on 1460 samples.
Best parameters: {'C': 10.0, 'epsilon': 0.05, 'gamma': 0.001, 'kernel': 'rbf'}
Submission saved to: submission.csv
     Id      SalePrice
0  1461  117755.992527
1  1462  154440.879469
2  1463  187213.415151
3  1464  198776.824405
4  1465  189834.488382
